In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from datetime import datetime
import os

# --- Core Logic Functions ---
def categorize_actor(actor_name):
    if pd.isna(actor_name): return 'Unidentified'
    name = str(actor_name).lower()
    if any(x in name for x in ['military forces of myanmar', 'police forces of myanmar', 'state administration council', 'border guard force', 'people\'s militia force']):
        return 'State Forces'
    elif any(x in name for x in ['pyu saw htee', 'thway thauk', 'blood comrades', 'swan arr shin', 'pro-military']):
        return 'Pro-Junta Militia'
    resistance_keywords = ['pdf', 'people\'s defense force', 'local defense force', 'kndf', 'karenni nationalities defense force', 'chinland defense force', 'cdf', 'unidentified anti-coup armed group', 'guerrilla', 'ogre', 'revolution', 'defense force', 'resistance', 'pafd', 'yaw defense force', 'ydf', 'dragon army', 'mrda', 'chindwin attack force', 'myingyan black tiger', 'mbt', 'black eagle', 'underground warriors', 'young force', 'ug-force', 'guerrilla force', 'defense team', 'ddt', 'special task force', 'sstf', 'attack force', 'generation z', 'gen z', 'drone strike', 'mdds', 'dark shadow', 'leopard army', 'blpa', 'chindwin brothers', 'taung nyo', 'eagle force', 'brave heart', 'danger force', 'red bandana', 'phoenix sgg', 'freeland attack', 'fla', 'support organization', 'commando', 'special force', 'vanguard', 'victory force', 'justice force', 'strike force', 'task force', 'people\'s army', 'liberation army', 'bha', 'tpf', 'kpaf', 'mmu', 'kso', 'sgg', 'baf', 'column', 'urban', 'cgm', 'technical team', 'shar htoo waw', 'king cobra', 'defence force', 'defence team', 'militia', 'security force', 'black k', 'thu rain', 'freedom force', 'pdaf', 'galon force', 'federal army', 'tiger force', 'ranger group', 'truth army', 'anonymous', 'oak awe', 'snake eyes']
    if any(x in name for x in resistance_keywords):
        return 'Resistance'
    eao_keywords = ['knu', 'knla', 'kndo', 'kia', 'kio', 'tnla', 'pslf', 'mndaa', 'mntjp', 'aa ', 'ula', 'rcss', 'ssa', 'knpp', 'ka ', 'cnp', 'sspp', 'pnlo', 'sna', 'cnf', 'cna', 'pno', 'pna', 'brotherhood alliance', 'northern alliance', 'three brotherhood', 'absdf', 'knlp', 'kpc', 'dkba', 'mnda', 'alp', 'ala', 'nssaa', 'shanni', 'kachin', 'karen', 'shan state', 'arakan', 'mon state', 'nmsp', 'nmla', 'chin brotherhood', 'rohingya', 'arsa', 'kaw thoo lei', 'pa-oh', 'ta\'ang', 'palaung', 'kokang', 'wa state', 'uwsa', 'mon national', 'naga', 'nscn', 'ktla']
    if any(x in name for x in eao_keywords):
        return 'EAOs'
    if 'protesters' in name: return 'Protesters'
    elif 'rioters' in name: return 'Rioters'
    elif 'civilians' in name: return 'Civilians'
    if 'unidentified' in name: return 'Unidentified'
    else: return 'Other Groups'

def extract_keywords(text_series, top_n=20):
    from collections import Counter
    stopwords = set(['the', 'and', 'a', 'to', 'in', 'of', 'on', 'with', 'for', 'at', 'by', 'from', 'an', 'is', 'was', 'were', 'it', 'as', 'that', 'this', 'reported', 'took', 'place', 'around', 'near', 'village', 'township', 'district', 'region', 'state', 'myanmar', 'burma', 'forces', 'military', 'junta', 'people', 'defence', 'force', 'pdf', 'tatmadaw', 'knu', 'kia', 'pdf', 'ldf', 'eao', 'ia', 'army'])
    all_words = ' '.join(text_series.fillna('').str.lower().replace(r'[^a-zA-Z\s]', '', regex=True)).split()
    filtered_words = [word for word in all_words if word not in stopwords and len(word) > 3]
    counts = Counter(filtered_words).most_common(top_n)
    return pd.DataFrame(counts, columns=['Keyword', 'Frequency'])

def extract_health_impacts(df):
    text_series = df['notes'].fillna('').str.lower()
    event_series = df['sub_event_type'].fillna('').str.lower()
    health_keywords = [r'hospital', r'clinic', r'medical', r'doctor', r'nurse', r'health', r'ambulance', r'medicine', r'patient', r'pharmacy', r'red cross', r'world health organization', r'who-led', r'unicef', r'displacement', r'malnutrition', r'injury', r'wounded', r'healthcare', r'sanitation', r'vaccination', r'epidemic', r'airstrike near hospital', r'shelling near hospital']
    pattern = '|'.join(health_keywords)
    health_hits = text_series.str.contains(pattern, case=False, regex=True)
    wellbeing_events = ['sexual violence', 'arrests', 'abduction', 'looting', 'property destruction']
    event_hits = event_series.apply(lambda x: any(e in x for e in wellbeing_events))
    return health_hits | event_hits


In [ ]:
# --- Data Loading & Preprocessing ---
DATA_PATH = '../data/ACLED Data Feb 14 2026.csv'
if not os.path.exists(DATA_PATH):
    print(f'Error: Data file not found at {DATA_PATH}')
else:
    df = pd.read_csv(DATA_PATH)
    df['event_date'] = pd.to_datetime(df['event_date'])
    df = df[df['event_date'] >= '2021-02-01']
    df['year_month'] = df['event_date'].dt.strftime('%Y-%m')
    df['actor1_clean'] = df['actor1'].apply(categorize_actor)
    df['actor2_clean'] = df['actor2'].apply(categorize_actor)

    node_color_map = {
        'State Forces': '#1e293b', 'Pro-Junta Militia': '#991b1b', 'Resistance': '#10b981', 
        'EAOs': '#0369a1', 'Civilians': '#f59e0b', 'Protesters': '#8b5cf6', 
        'Rioters': '#f97316', 'Unidentified': '#475569', 'Other Groups': '#64748b'
    }
    print(f'Data loaded: {len(df)} events.')


In [ ]:
# --- Early Warning Analytics (Z-Score & Projection) ---
# 1. Z-Score Anomaly Detection (Last 90 Days)
df_90 = df[df['event_date'] > (df['event_date'].max() - pd.Timedelta(days=90))]
weekly_counts = df_90.groupby(['admin1', pd.Grouper(key='event_date', freq='W')]).size().reset_index(name='count')

anomalies = []
for region in weekly_counts['admin1'].unique():
    reg_data = weekly_counts[weekly_counts['admin1'] == region]['count']
    if len(reg_data) > 4:
        mean, std = reg_data.mean(), reg_data.std()
        current = reg_data.iloc[-1]
        z_score = (current - mean) / std if std > 0 else 0
        if z_score > 1.5:
            anomalies.append({'Region': region, 'Z-Score': round(z_score, 2), 'Status': 'Surge Detected'})

print('High-Risk Anomalies (Last 7 Days):')
print(pd.DataFrame(anomalies))

# 2. Linear Regression (Next 7 Days Projection)
df_trend = df[df['event_date'] > (df['event_date'].max() - pd.Timedelta(days=28))]
weekly_fats = df_trend.groupby(pd.Grouper(key='event_date', freq='W'))['fatalities'].sum().reset_index()

if len(weekly_fats) >= 2:
    x = np.arange(len(weekly_fats))
    y = weekly_fats['fatalities'].values
    slope, intercept = np.polyfit(x, y, 1)
    projection = max(0, slope * (len(weekly_fats)) + intercept)
    
    fig_proj = go.Figure()
    fig_proj.add_trace(go.Scatter(x=weekly_fats['event_date'], y=y, name='Historical fatalities', line=dict(color='#94a3b8')))
    future_date = weekly_fats['event_date'].iloc[-1] + pd.Timedelta(days=7)
    fig_proj.add_trace(go.Scatter(x=[weekly_fats['event_date'].iloc[-1], future_date], y=[y[-1], projection], name='7-Day Projection', line=dict(color='#10b981', dash='dash')))
    fig_proj.update_layout(title='Fatality Trajectory & Risk Projection', paper_bgcolor='white', plot_bgcolor='white')
    fig_proj.show()

# 3. Risk Matrix
risk_df = df.groupby('admin1').agg({'event_id_cnty': 'count','fatalities': 'sum'}).rename(columns={'event_id_cnty': 'Frequency'})
risk_df['Lethality'] = (risk_df['fatalities'] / risk_df['Frequency']).round(2)
fig_matrix = px.scatter(risk_df.reset_index(), x='Frequency', y='Lethality', text='admin1', size='fatalities', color='Lethality', color_continuous_scale='Reds')
fig_matrix.update_traces(textposition='top center')
fig_matrix.add_hline(y=risk_df['Lethality'].mean(), line_dash='dash', annotation_text='Baseline Lethality')
fig_matrix.update_layout(title='Regional Risk Matrix (SDG 3.D Early Warning)', paper_bgcolor='white', plot_bgcolor='white')
fig_matrix.show()
